Bootstrap

In [11]:
from _notebook_bootstrap import bootstrap

repo_root, datasets_root = bootstrap()
repo_root, datasets_root

(PosixPath('/home/user/workspace/datasets-utilities'),
 PosixPath('/home/user/data/datasets'))

Imports

In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
import copy
import pypowsybl as pp
from scipy.io import loadmat
from pypower.api import rundcpf, runpf, ppoption

from dataloaders.rte7000_opensynth import Rte7000OpenSynth

Load dataset object

In [13]:
# Select your file (can be changed to any other snapshot path under datasets_root)
xiidm_relpath = "d_gitt_rte7000_2021/rte-01Jan21-0000.xiidm"

ds = Rte7000OpenSynth(datasets_root=datasets_root, relpath=xiidm_relpath)
xiidm_file = ds.path

Convert to matpower

In [14]:
#input_file = Path("./data/rte-01Jan21-0000.xiidm.bz2").resolve()

out_dir = Path("./converted").resolve()
out_dir.mkdir(exist_ok=True)

# For PyPowSyBl MATPOWER export, use .mat
out_file = out_dir / "rte-01Jan21-0000.mat"

grid = pp.network.load(xiidm_file)
grid.save(out_file, format="MATPOWER")

print("Exported:", out_file)
print("Exists:", out_file.exists())
print("Size:", out_file.stat().st_size, "bytes")

Exported: /home/user/workspace/datasets-utilities/notebooks/converted/rte-01Jan21-0000.mat
Exists: True
Size: 283757 bytes


In [15]:
out_file = Path("./converted/rte-01Jan21-0000.mat").resolve()

mat = loadmat(out_file, squeeze_me=True, struct_as_record=False)

print("Top-level keys:")
print([k for k in mat.keys() if not k.startswith("__")])

Top-level keys:
['mpc']


In [16]:
out_file = Path("./converted/rte-01Jan21-0000.mat").resolve()

mat = loadmat(out_file, squeeze_me=True, struct_as_record=False)

# MATPOWER .mat files usually contain variable "mpc"
if "mpc" not in mat:
    raise KeyError(f"No 'mpc' variable found. Available keys: {[k for k in mat.keys() if not k.startswith('__')]}")

mpc = mat["mpc"]

required = ["baseMVA", "bus", "gen", "branch"]

for field in required:
    if not hasattr(mpc, field):
        raise AttributeError(f"Missing MATPOWER field: {field}")

print("baseMVA:", mpc.baseMVA)
print("bus shape:", mpc.bus.shape)
print("gen shape:", mpc.gen.shape)
print("branch shape:", mpc.branch.shape)

if hasattr(mpc, "gencost"):
    print("gencost shape:", mpc.gencost.shape)
else:
    print("No gencost field")

baseMVA: 100.0
bus shape: (6404, 13)
gen shape: (2242, 21)
branch shape: (8891, 13)
No gencost field


In [17]:
matpower_file = Path("./converted/rte-01Jan21-0000.mat").resolve()
mat = loadmat(matpower_file, squeeze_me=True, struct_as_record=False)
mpc = mat["mpc"]

ppc = {
    "version": "2",
    "baseMVA": float(np.asarray(mpc.baseMVA).squeeze()),
    "bus": np.asarray(mpc.bus, dtype=float),
    "gen": np.asarray(mpc.gen, dtype=float),
    "branch": np.asarray(mpc.branch, dtype=float),
}

bus_cols = [
    "BUS_I", "BUS_TYPE", "PD", "QD", "GS", "BS", "BUS_AREA",
    "VM", "VA", "BASE_KV", "ZONE", "VMAX", "VMIN"
]

gen_cols = [
    "GEN_BUS", "PG", "QG", "QMAX", "QMIN", "VG", "MBASE",
    "GEN_STATUS", "PMAX", "PMIN", "PC1", "PC2", "QC1MIN",
    "QC1MAX", "QC2MIN", "QC2MAX", "RAMP_AGC", "RAMP_10",
    "RAMP_30", "RAMP_Q", "APF"
]

branch_cols = [
    "F_BUS", "T_BUS", "BR_R", "BR_X", "BR_B",
    "RATE_A", "RATE_B", "RATE_C", "TAP", "SHIFT",
    "BR_STATUS", "ANGMIN", "ANGMAX"
]

def nonfinite_report(name, arr, cols):
    arr = np.asarray(arr, dtype=float)
    mask = ~np.isfinite(arr)

    print(f"\n{name}")
    print("shape:", arr.shape)
    print("non-finite cells:", int(mask.sum()))
    print("rows with any non-finite:", int(mask.any(axis=1).sum()))

    counts = mask.sum(axis=0)
    report = pd.DataFrame({
        "col_index": range(arr.shape[1]),
        "column": cols[:arr.shape[1]],
        "nonfinite_count": counts
    })

    display(report[report["nonfinite_count"] > 0])

nonfinite_report("bus", ppc["bus"], bus_cols)
nonfinite_report("gen", ppc["gen"], gen_cols)
nonfinite_report("branch", ppc["branch"], branch_cols)

print("\nBus type counts:")
print(pd.Series(ppc["bus"][:, 1]).value_counts(dropna=False).sort_index())

print("\nGenerator status counts:")
print(pd.Series(ppc["gen"][:, 7]).value_counts(dropna=False).sort_index())

print("\nBranch status counts:")
print(pd.Series(ppc["branch"][:, 10]).value_counts(dropna=False).sort_index())


bus
shape: (6404, 13)
non-finite cells: 7382
rows with any non-finite: 3691


,col_index,column,nonfinite_count
2,2,PD,3691
3,3,QD,3691



gen
shape: (2242, 21)
non-finite cells: 2235
rows with any non-finite: 2235


,col_index,column,nonfinite_count
1,1,PG,2235



branch
shape: (8891, 13)
non-finite cells: 0
rows with any non-finite: 0


,col_index,column,nonfinite_count



Bus type counts:
1.0    5934
2.0     469
3.0       1
Name: count, dtype: int64

Generator status counts:
1.0    2242
Name: count, dtype: int64

Branch status counts:
1.0    8891
Name: count, dtype: int64


In [18]:
matpower_file = Path("./converted/rte-01Jan21-0000.mat").resolve()

mat = loadmat(matpower_file, squeeze_me=True, struct_as_record=False)
mpc = mat["mpc"]

ppc = {
    "version": "2",
    "baseMVA": float(np.asarray(mpc.baseMVA).squeeze()),
    "bus": np.asarray(mpc.bus, dtype=float),
    "gen": np.asarray(mpc.gen, dtype=float),
    "branch": np.asarray(mpc.branch, dtype=float),
}

ppc_clean = copy.deepcopy(ppc)

bus = ppc_clean["bus"]
gen = ppc_clean["gen"]
branch = ppc_clean["branch"]

# MATPOWER / PYPOWER column indices
PD = 2
QD = 3
PG = 1

print("Before cleaning:")
print("NaN PD:", np.isnan(bus[:, PD]).sum())
print("NaN QD:", np.isnan(bus[:, QD]).sum())
print("NaN PG:", np.isnan(gen[:, PG]).sum())

# Preserve known values, replace only missing injections
bus[:, PD] = np.nan_to_num(bus[:, PD], nan=0.0)
bus[:, QD] = np.nan_to_num(bus[:, QD], nan=0.0)
gen[:, PG] = np.nan_to_num(gen[:, PG], nan=0.0)

print("\nAfter cleaning:")
print("NaN PD:", np.isnan(bus[:, PD]).sum())
print("NaN QD:", np.isnan(bus[:, QD]).sum())
print("NaN PG:", np.isnan(gen[:, PG]).sum())

print("\nTotals after cleaning:")
print("Total PD:", bus[:, PD].sum())
print("Total QD:", bus[:, QD].sum())
print("Total PG:", gen[:, PG].sum())

print("\nAll finite:")
print("bus:", np.isfinite(bus).all())
print("gen:", np.isfinite(gen).all())
print("branch:", np.isfinite(branch).all())

Before cleaning:
NaN PD: 3691
NaN QD: 3691
NaN PG: 2235

After cleaning:
NaN PD: 0
NaN QD: 0
NaN PG: 0

Totals after cleaning:
Total PD: 0.0
Total QD: 0.0
Total PG: 0.0

All finite:
bus: True
gen: True
branch: True


In [19]:
ppopt_dc = ppoption(
    VERBOSE=2,
    OUT_ALL=1
)

dc_results, dc_success = rundcpf(ppc_clean, ppopt_dc)

print("DC PF success:", dc_success)

bus_res = dc_results["bus"]
branch_res = dc_results["branch"]

print("finite bus results:", np.isfinite(bus_res).all())
print("finite branch results:", np.isfinite(branch_res).all())

print("Va min/max:", np.nanmin(bus_res[:, 8]), np.nanmax(bus_res[:, 8]))

if branch_res.shape[1] >= 17:
    print("Pf min/max:", np.nanmin(branch_res[:, 13]), np.nanmax(branch_res[:, 13]))

PYPOWER Version 5.1.18, 10-Apr-2025 -- DC Power Flow

Converged in 0.02 seconds
|     System Summary                                                           |

How many?                How much?              P (MW)            Q (MVAr)
---------------------    -------------------  -------------  -----------------
Buses           6404     Total Gen Capacity   92827.7           0.0 to 0.0
Generators      2238     On-line Capacity     92827.7           0.0 to 0.0
Committed Gens  2238     Generation (actual)     -0.0               0.0
Loads              4     Load                     0.0               0.0
  Fixed            0       Fixed                  0.0               0.0
  Dispatchable     4       Dispatchable          -0.0 of 32.7      -0.0
Shunts             0     Shunt (inj)              0.0               0.0
Branches        8891     Losses (I^2 * Z)         0.00              0.00
Transformers    1677     Branch Charging (inj)     -                0.0
Inter-ties         0     Tota

In [20]:
ppopt_ac = ppoption(
    VERBOSE=2,
    OUT_ALL=1,
    PF_ALG=1,
    PF_MAX_IT=100,
    PF_TOL=1e-6
)

ac_results, ac_success = runpf(ppc_clean, ppopt_ac)

print("AC PF success:", ac_success)

bus_res = ac_results["bus"]
branch_res = ac_results["branch"]

print("finite bus results:", np.isfinite(bus_res).all())
print("finite branch results:", np.isfinite(branch_res).all())

print("Vm min/max:", np.nanmin(bus_res[:, 7]), np.nanmax(bus_res[:, 7]))
print("Va min/max:", np.nanmin(bus_res[:, 8]), np.nanmax(bus_res[:, 8]))

if branch_res.shape[1] >= 17:
    print("Pf min/max:", np.nanmin(branch_res[:, 13]), np.nanmax(branch_res[:, 13]))
    print("Qf min/max:", np.nanmin(branch_res[:, 14]), np.nanmax(branch_res[:, 14]))

PYPOWER Version 5.1.18, 10-Apr-2025 -- AC Power Flow (Newton)


 it    max P & Q mismatch (p.u.)
----  ---------------------------
  0         4.903e+02
  1         3.075e+01
  2         9.157e+00
  3         3.244e-01


  4         5.176e-04
  5         1.422e-09
Newton's method power flow converged in 5 iterations.

Converged in 0.21 seconds
|     System Summary                                                           |

How many?                How much?              P (MW)            Q (MVAr)
---------------------    -------------------  -------------  -----------------
Buses           6404     Total Gen Capacity   92827.7       -27556.3 to 29622.7
Generators      2238     On-line Capacity     92827.7       -27556.3 to 29622.7
Committed Gens  2238     Generation (actual)    125.7           -20421.1
Loads              4     Load                     0.0              13.5
  Fixed            0       Fixed                  0.0               0.0
  Dispatchable     4       Dispatchable          -0.0 of 32.7      13.5
Shunts           123     Shunt (inj)             -0.0           -3549.4
Branches        8891     Losses (I^2 * Z)       125.68           1547.75
Transformers    1677     Branch Charging (in